# 6. Hyperparameter Tuning

In [1]:
import os
import numpy as np
import pandas as pd

from scipy.sparse import load_npz

# Load TF-IDF features
X_train = load_npz("../data/processed/X_train_tfidf.npz")
X_val = load_npz("../data/processed/X_val_tfidf.npz")
X_test = load_npz("../data/processed/X_test_tfidf.npz")

# Load labels
y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Test features:", X_test.shape)

print("\nTraining labels:", y_train.shape)
print("Validation labels:", y_val.shape)
print("Test labels:", y_test.shape)

print("\nFeature matrix type:", type(X_train))

Training features: (15999, 32992)
Validation features: (2000, 32992)
Test features: (2000, 32992)

Training labels: (15999,)
Validation labels: (2000,)
Test labels: (2000,)

Feature matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [2]:
#  Linear SVM Tuning Baseline

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

baseline_svm = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=2000
)

baseline_svm.fit(X_train, y_train)

y_val_pred_baseline = baseline_svm.predict(X_val)

baseline_svm_accuracy = accuracy_score(
    y_val,
    y_val_pred_baseline
)

baseline_svm_macro_f1 = f1_score(
    y_val,
    y_val_pred_baseline,
    average="macro"
)

baseline_svm_weighted_f1 = f1_score(
    y_val,
    y_val_pred_baseline,
    average="weighted"
)

print(f"Baseline Accuracy: {baseline_svm_accuracy:.4f}")
print(f"Baseline Macro F1: {baseline_svm_macro_f1:.4f}")
print(f"Baseline Weighted F1: {baseline_svm_weighted_f1:.4f}")

Baseline Accuracy: 0.9030
Baseline Macro F1: 0.8735
Baseline Weighted F1: 0.9032


In [3]:
#  Linear SVM Hyperparameter Grid

from sklearn.model_selection import ParameterGrid

svm_param_grid = {
    "C": [0.25, 0.5, 1.0, 2.0],
    "class_weight": [None, "balanced"]
}

svm_combinations = list(ParameterGrid(svm_param_grid))

print("Total parameter combinations:", len(svm_combinations))

for i, params in enumerate(svm_combinations, start=1):
    print(f"{i}. {params}")

Total parameter combinations: 8
1. {'C': 0.25, 'class_weight': None}
2. {'C': 0.25, 'class_weight': 'balanced'}
3. {'C': 0.5, 'class_weight': None}
4. {'C': 0.5, 'class_weight': 'balanced'}
5. {'C': 1.0, 'class_weight': None}
6. {'C': 1.0, 'class_weight': 'balanced'}
7. {'C': 2.0, 'class_weight': None}
8. {'C': 2.0, 'class_weight': 'balanced'}


In [4]:
#  Linear SVM Hyperparameter Search

svm_results = []

for params in svm_combinations:
    model = LinearSVC(
        C=params["C"],
        class_weight=params["class_weight"],
        random_state=42,
        max_iter=2000
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    weighted_f1 = f1_score(y_val, y_pred, average="weighted")

    svm_results.append({
        "C": params["C"],
        "class_weight": params["class_weight"],
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

svm_results_df = pd.DataFrame(svm_results)

svm_results_df = svm_results_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

print(svm_results_df.to_string(index=False))

   C class_weight  Accuracy  Macro F1  Weighted F1
0.50     balanced    0.9035  0.877315     0.904086
0.25     balanced    0.9025  0.875863     0.903234
1.00     balanced    0.9030  0.873497     0.903208
2.00     balanced    0.9020  0.870881     0.902089
2.00          NaN    0.8980  0.863662     0.897017
1.00          NaN    0.8960  0.860540     0.894777
0.50          NaN    0.8920  0.856002     0.890302
0.25          NaN    0.8840  0.849593     0.881893


In [6]:
from sklearn.metrics import classification_report


In [7]:
# Evaluating the Best Linear SVM Configuration

best_svm_model = LinearSVC(
    C=0.5,
    class_weight="balanced",
    random_state=42,
    max_iter=2000
)

best_svm_model.fit(X_train, y_train)

y_val_pred_best_svm = best_svm_model.predict(X_val)

print("Best Linear SVM")
print("C:", best_svm_model.C)
print("Class weight:", best_svm_model.class_weight)

print(f"\nValidation Accuracy: {accuracy_score(y_val, y_val_pred_best_svm):.4f}")
print(
    f"Validation Macro F1: "
    f"{f1_score(y_val, y_val_pred_best_svm, average='macro'):.4f}"
)
print(
    f"Validation Weighted F1: "
    f"{f1_score(y_val, y_val_pred_best_svm, average='weighted'):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_best_svm,
        target_names=[
            "sadness",
            "joy",
            "love",
            "anger",
            "fear",
            "surprise"
        ],
        zero_division=0
    )
)

Best Linear SVM
C: 0.5
Class weight: balanced

Validation Accuracy: 0.9035
Validation Macro F1: 0.8773
Validation Weighted F1: 0.9041

Classification Report:
              precision    recall  f1-score   support

     sadness       0.94      0.93      0.93       550
         joy       0.93      0.92      0.92       704
        love       0.78      0.89      0.83       178
       anger       0.92      0.89      0.91       275
        fear       0.87      0.84      0.86       212
    surprise       0.78      0.84      0.81        81

    accuracy                           0.90      2000
   macro avg       0.87      0.89      0.88      2000
weighted avg       0.91      0.90      0.90      2000



In [8]:
#  Logistic Regression Hyperparameter Grid

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid

logreg_param_grid = {
    "C": [0.25, 0.5, 1.0, 2.0],
    "class_weight": [None, "balanced"]
}

logreg_combinations = list(ParameterGrid(logreg_param_grid))

print("Total parameter combinations:", len(logreg_combinations))

for i, params in enumerate(logreg_combinations, start=1):
    print(f"{i}. {params}")

Total parameter combinations: 8
1. {'C': 0.25, 'class_weight': None}
2. {'C': 0.25, 'class_weight': 'balanced'}
3. {'C': 0.5, 'class_weight': None}
4. {'C': 0.5, 'class_weight': 'balanced'}
5. {'C': 1.0, 'class_weight': None}
6. {'C': 1.0, 'class_weight': 'balanced'}
7. {'C': 2.0, 'class_weight': None}
8. {'C': 2.0, 'class_weight': 'balanced'}


In [9]:
#  Logistic Regression Hyperparameter Search

logreg_results = []

for params in logreg_combinations:
    model = LogisticRegression(
        C=params["C"],
        class_weight=params["class_weight"],
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    weighted_f1 = f1_score(y_val, y_pred, average="weighted")

    logreg_results.append({
        "C": params["C"],
        "class_weight": params["class_weight"],
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

logreg_results_df = pd.DataFrame(logreg_results)

logreg_results_df = logreg_results_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

print(logreg_results_df.to_string(index=False))

   C class_weight  Accuracy  Macro F1  Weighted F1
2.00     balanced    0.8865  0.857939     0.887296
1.00     balanced    0.8755  0.848787     0.876498
0.50     balanced    0.8610  0.837001     0.862234
2.00          NaN    0.8625  0.814141     0.857968
0.25     balanced    0.8285  0.805802     0.830367
1.00          NaN    0.8300  0.751094     0.819526
0.50          NaN    0.7615  0.622391     0.734619
0.25          NaN    0.6735  0.442701     0.610767


## Selecting the Final Model

Based on validation performance, the tuned Linear SVM is selected as the final candidate model.

The selected configuration is:

- Model: Linear SVM
- C: 0.5
- Class weight: balanced
- Validation Accuracy: 0.9035
- Validation Macro F1: 0.8773
- Validation Weighted F1: 0.9041

The test dataset has not been used for model selection.

In [10]:
import joblib

final_model = LinearSVC(
    C=0.5,
    class_weight="balanced",
    random_state=42,
    max_iter=2000
)

final_model.fit(X_train, y_train)

model_path = "../models/final_svm_model.pkl"

joblib.dump(final_model, model_path)

print("Final model saved:", model_path)
print("File exists:", os.path.exists(model_path))

Final model saved: ../models/final_svm_model.pkl
File exists: True


##  Hyperparameter Tuning Summary

Hyperparameter tuning was performed for Linear SVM and Logistic Regression using the validation dataset.

Linear SVM achieved the strongest validation performance.

The selected final candidate is:

- Model: Linear SVM
- C: 0.5
- Class weight: balanced
- Validation Accuracy: 0.9035
- Validation Macro F1: 0.8773
- Validation Weighted F1: 0.9041

The tuned Linear SVM outperformed the tuned Logistic Regression model and the original baseline models.

The final Linear SVM model was saved as:

`models/final_svm_model.pkl`

The test dataset was not used during hyperparameter tuning or model selection. It will be used only for the final evaluation in the model evaluation notebook.